# Raag Classification Project – 02 Cleaning Pipeline

**Goal of this notebook**

Turn the raw Gurbani text into a clean, normalized text string that is ready for:
- Raag heading extraction
- Downstream ML tasks (later)

**Inputs**

- Raw text produced by the `pdf_loader` production function.

**Outputs**

- A cleaned text variable (e.g. `cleaned_gurbani_text`).
- A saved file (e.g. `data/processed/cleaned_gurbani_text.txt`) for later steps.

**Cleaning rules (from exploration)**

- Normalize line breaks (`\r\n`, `\r` → `\n`).
- Remove zero-width characters (e.g. `\ufeff`, `\u200b` etc.).
- Collapse multiple spaces into a single space.
- Collapse 3+ blank lines into max 2.
- Trim leading/trailing whitespace.

This notebook is the *bridge* between raw data and structured analysis.


## 1. Imports and setup

## 2. Load 1st page from pdf which has the index of Raag names

In [51]:
import pdfplumber
pdf_file_path = "../data/raw_text/Gurbani_with_index_Uni.pdf"

with pdfplumber.open(pdf_file_path) as pdf:
    gurbani_page1 = pdf.pages[0]
    gurbani_page1_text = gurbani_page1.extract_text()
len(gurbani_page1_text)

2182

In [33]:
gurbani_page1_text

'1\n❁❁❁❁❁❁❁❁❁❁❁❁❁❁ ❁❁❁❁❁❁❁❁❁❁❁❁❁❁\n❁\n❁\n❁\n❁ ੧ਓ ਸਤਿਗੁਰ ਪ੍ਰਸਾਤਿ ॥ ਰਾਗੁ ਟੋਡੀ ............................... ੭੧੧ ਰਾਗੁ ਪ੍ਰਭਾਿੀ ........................ ੧੩੨੭\n❁\n❁ ਰਾਗੁ ਿੈਰਾੜੀ ............................੭੧੯ ਰਾਗੁ ਜੈਜਾਵੰਿੀ ...................... ੧੩੫੨\nਿਿਕਰਾ ਰਾਗਾਂ ਕਾ\n❁\n❁ ਰਾਗੁ ਤਿਲੰਗ ............................ ੭੨੧ ਸਲੋਕ ਸਿਸਤਿਿੀ ਮਿਲਾ ੧ .... ੧੩੫੩\nਰਾਗ ........................................ ਪ੍ੰਨਾ ❁\n❁ ਰਾਗੁ ਸੂਿੀ .............................. ੭੨੮ ਸਲੋਕ ਸਿਸਤਿਿੀ ਮਿਲਾ ੫ .... ੧੩੫੩\nਜਪ੍ੁ ........................................... ੧\n❁\n❁ ਰਾਗੁ ਤਿਲਾਵਲੁ ........................ ੭੯੫ ਮਿਲਾ ੫ ਗਾਥਾ .....................੧੩੬੦\nਸੋ ਿਰੁ ........................................ ੮\n❁\n❁ ਰਾਗੁ ਗੋੋਂਡ .............................. ੮੫੯ ਫੁਨਿੇ ਮਿਲਾ ੫ ..................... ੧੩੬੧\nਸੋ ਪ੍ੁਰਖੁ ................................... ੧੦\n❁\n❁ ਰਾਗੁ ਰਾਮਕਲੀ ......................... ੮੭੬ ਚਉਿੋਲੇ ਮਿਲਾ ੫ .................. ੧੩੬੩\nਸੋਤਿਲਾ ................................... ੧੨\n❁\n❁ ਰਾਗੁ ਨਟ ਨਾਰਾਇਨ ................. ੯੭੫ ਸਲੋਕ ਭਗਿ ਕਿੀਰ ਜੀਉ 


## 3. Define cleaning steps (high-level)

Cleaning Steps:
* Remove decorative symbols.
* Remove dotted leader lines.
* Normalize newlines
* Collapse repeated spaces/weird spacing.
* Strip Zero-width/invisible characters if present.

**We will use regex to clean up our string and extract the 31 raag names**

## 4. Official label list for Raag labels

In [50]:
RAAG_MASTER_LIST = [
     "ਸਿਰੀ ਰਾਗੁ", "ਰਾਗੁ ਮਾਝ", "ਰਾਗੁ ਗਉੜੀ", "ਰਾਗੁ ਆਸਾ", "ਰਾਗੁ ਗੂਜਰੀ", "ਰਾਗੁ ਦੇਵਗੰਧਾਰੀ", "ਰਾਗੁ ਬਿਹਾਗੜਾ", 
    "ਰਾਗੁ ਵਡਹੰਸ", "ਰਾਗੁ ਸੋਰਠਿ", "ਰਾਗੁ ਧਨਾਸਰੀ", "ਰਾਗੁ ਜੈਤਸਰੀ", "ਰਾਗੁ ਟੋਡੀ", "ਰਾਗੁ ਬੈਰਾੜੀ", "ਰਾਗੁ ਤਿਲੰਗ", 
    "ਰਾਗੁ ਸੂਹੀ", "ਰਾਗੁ ਬਿਲਾਵਲ", "ਰਾਗੁ ਗੋਂਡ", "ਰਾਗੁ ਰਾਮਕਲੀ", "ਰਾਗੁ ਨਟ ਨਾਰਾਇਨ", "ਰਾਗੁ ਮਾਲੀ ਗਉੜਾ", "ਰਾਗੁ ਮਾਰੂ", 
    "ਰਾਗੁ ਤੁਖਾਰੀ", "ਰਾਗੁ ਕੇਦਾਰਾ", "ਰਾਗੁ ਭੈਰਉ", "ਰਾਗੁ ਬਸੰਤੁ", "ਰਾਗੁ ਸਾਰਗ", "ਰਾਗੁ ਮਲਾਰ", "ਰਾਗੁ ਕਾਨੜਾ", 
    "ਰਾਗੁ ਕਲਿਆਨ", "ਰਾਗੁ ਪ੍ਰਭਾਤੀ", "ਰਾਗੁ ਜੈਜਾਵੰਤੀ"
]

print(f"the total number of Raags in the Raag Master List is {len(RAAG_MASTER_LIST)}.")

the total number of Raags in the Raag Master List is 31.


## 5. Apply cleaning steps

In [34]:
import re
# make a copy of txt string to apply regex
gurbani_page1_copy = gurbani_page1_text

# use regex pattern to extract pattern in gurmukhi, use the Unicode block range [\u0A00-\u0A7F]
raag_names_pattern = re.compile(r"(ਰਾਗੁ\s+[\u0A00-\u0A7F]+|[\u0A00-\u0A7F]+\s+ਰਾਗੁ)")
raag_matches = raag_names_pattern.findall(gurbani_page1_copy)
len(raag_matches)

31

In [35]:
raag_matches

['ਰਾਗੁ ਟੋਡੀ',
 '੭੧੧ ਰਾਗੁ',
 'ਰਾਗੁ ਿੈਰਾੜੀ',
 '੭੧੯ ਰਾਗੁ',
 'ਰਾਗੁ ਤਿਲੰਗ',
 'ਰਾਗੁ ਸੂਿੀ',
 'ਰਾਗੁ ਤਿਲਾਵਲੁ',
 'ਰਾਗੁ ਗੋੋਂਡ',
 'ਰਾਗੁ ਰਾਮਕਲੀ',
 'ਰਾਗੁ ਨਟ',
 'ਤਸਰੀ ਰਾਗੁ',
 'ਰਾਗੁ ਮਾਲੀ',
 '੧੩੭੭\nਰਾਗੁ',
 'ਰਾਗੁ ਮਾਰੂ',
 '੧੩੮੫\nਰਾਗੁ',
 'ਰਾਗੁ ਿੁਖਾਰੀ',
 '੧੪੧੦\nਰਾਗੁ',
 'ਰਾਗੁ ਕੇਿਾਰਾ',
 '੧੪੨੬\nਰਾਗੁ',
 'ਰਾਗੁ ਭੈਰਉ',
 '੧੪੨੯\nਰਾਗੁ',
 'ਰਾਗੁ ਿਸੰਿੁ',
 '੧੪੨੯\nਰਾਗੁ',
 'ਰਾਗੁ ਸਾਰਗ',
 'ਰਾਗੁ ਵਡਿੰਸੁ',
 'ਰਾਗੁ ਮਲਾਰ',
 'ਰਾਗੁ ਸੋਰਤਿ',
 'ਰਾਗੁ ਕਾਨੜਾ',
 'ਰਾਗੁ ਧਨਾਸਰੀ',
 'ਰਾਗੁ ਕਤਲਆਨ',
 'ਰਾਗੁ ਜੈਿਸਰੀ']

In [38]:
raag_names_pattern_2 = re.compile(r"ਰਾਗੁ\s+[\u0A00-\u0A7F]+")

raag_matches_2 = raag_names_pattern_2.findall(gurbani_page1_copy)
len(raag_matches_2)

30

In [39]:
raag_matches_2

['ਰਾਗੁ ਟੋਡੀ',
 'ਰਾਗੁ ਪ੍ਰਭਾਿੀ',
 'ਰਾਗੁ ਿੈਰਾੜੀ',
 'ਰਾਗੁ ਜੈਜਾਵੰਿੀ',
 'ਰਾਗੁ ਤਿਲੰਗ',
 'ਰਾਗੁ ਸੂਿੀ',
 'ਰਾਗੁ ਤਿਲਾਵਲੁ',
 'ਰਾਗੁ ਗੋੋਂਡ',
 'ਰਾਗੁ ਰਾਮਕਲੀ',
 'ਰਾਗੁ ਨਟ',
 'ਰਾਗੁ ਮਾਲੀ',
 'ਰਾਗੁ ਮਾਝ',
 'ਰਾਗੁ ਮਾਰੂ',
 'ਰਾਗੁ ਗਉੜੀ',
 'ਰਾਗੁ ਿੁਖਾਰੀ',
 'ਰਾਗੁ ਆਸਾ',
 'ਰਾਗੁ ਕੇਿਾਰਾ',
 'ਰਾਗੁ ਗੂਜਰੀ',
 'ਰਾਗੁ ਭੈਰਉ',
 'ਰਾਗੁ ਿੇਵਗੰਧਾਰੀ',
 'ਰਾਗੁ ਿਸੰਿੁ',
 'ਰਾਗੁ ਤਿਿਾਗੜਾ',
 'ਰਾਗੁ ਸਾਰਗ',
 'ਰਾਗੁ ਵਡਿੰਸੁ',
 'ਰਾਗੁ ਮਲਾਰ',
 'ਰਾਗੁ ਸੋਰਤਿ',
 'ਰਾਗੁ ਕਾਨੜਾ',
 'ਰਾਗੁ ਧਨਾਸਰੀ',
 'ਰਾਗੁ ਕਤਲਆਨ',
 'ਰਾਗੁ ਜੈਿਸਰੀ']

In [43]:
combined_raag_matches = set(raag_matches + raag_matches_2)
len(combined_raag_matches)

38

In [44]:
combined_raag_matches

{'ਤਸਰੀ ਰਾਗੁ',
 'ਰਾਗੁ ਆਸਾ',
 'ਰਾਗੁ ਕਤਲਆਨ',
 'ਰਾਗੁ ਕਾਨੜਾ',
 'ਰਾਗੁ ਕੇਿਾਰਾ',
 'ਰਾਗੁ ਗਉੜੀ',
 'ਰਾਗੁ ਗੂਜਰੀ',
 'ਰਾਗੁ ਗੋੋਂਡ',
 'ਰਾਗੁ ਜੈਜਾਵੰਿੀ',
 'ਰਾਗੁ ਜੈਿਸਰੀ',
 'ਰਾਗੁ ਟੋਡੀ',
 'ਰਾਗੁ ਤਿਲਾਵਲੁ',
 'ਰਾਗੁ ਤਿਲੰਗ',
 'ਰਾਗੁ ਤਿਿਾਗੜਾ',
 'ਰਾਗੁ ਧਨਾਸਰੀ',
 'ਰਾਗੁ ਨਟ',
 'ਰਾਗੁ ਪ੍ਰਭਾਿੀ',
 'ਰਾਗੁ ਭੈਰਉ',
 'ਰਾਗੁ ਮਲਾਰ',
 'ਰਾਗੁ ਮਾਝ',
 'ਰਾਗੁ ਮਾਰੂ',
 'ਰਾਗੁ ਮਾਲੀ',
 'ਰਾਗੁ ਰਾਮਕਲੀ',
 'ਰਾਗੁ ਵਡਿੰਸੁ',
 'ਰਾਗੁ ਸਾਰਗ',
 'ਰਾਗੁ ਸੂਿੀ',
 'ਰਾਗੁ ਸੋਰਤਿ',
 'ਰਾਗੁ ਿਸੰਿੁ',
 'ਰਾਗੁ ਿੁਖਾਰੀ',
 'ਰਾਗੁ ਿੇਵਗੰਧਾਰੀ',
 'ਰਾਗੁ ਿੈਰਾੜੀ',
 '੧੩੭੭\nਰਾਗੁ',
 '੧੩੮੫\nਰਾਗੁ',
 '੧੪੧੦\nਰਾਗੁ',
 '੧੪੨੬\nਰਾਗੁ',
 '੧੪੨੯\nਰਾਗੁ',
 '੭੧੧ ਰਾਗੁ',
 '੭੧੯ ਰਾਗੁ'}

In [47]:
# Filter out names that start with gurmukhi digits
gurmukhi_digits = "੦੧੨੩੪੫੬੭੮੯"
final_raag_matches = [match for match in combined_raag_matches if match[0] not in gurmukhi_digits]
len(final_raag_matches)

31

In [48]:
final_raag_matches

['ਰਾਗੁ ਤਿਿਾਗੜਾ',
 'ਰਾਗੁ ਗਉੜੀ',
 'ਰਾਗੁ ਮਾਰੂ',
 'ਰਾਗੁ ਤਿਲਾਵਲੁ',
 'ਰਾਗੁ ਗੂਜਰੀ',
 'ਰਾਗੁ ਗੋੋਂਡ',
 'ਰਾਗੁ ਮਾਲੀ',
 'ਰਾਗੁ ਕਤਲਆਨ',
 'ਰਾਗੁ ਟੋਡੀ',
 'ਰਾਗੁ ਿੇਵਗੰਧਾਰੀ',
 'ਰਾਗੁ ਿਸੰਿੁ',
 'ਰਾਗੁ ਿੈਰਾੜੀ',
 'ਰਾਗੁ ਜੈਜਾਵੰਿੀ',
 'ਰਾਗੁ ਤਿਲੰਗ',
 'ਰਾਗੁ ਵਡਿੰਸੁ',
 'ਤਸਰੀ ਰਾਗੁ',
 'ਰਾਗੁ ਮਾਝ',
 'ਰਾਗੁ ਰਾਮਕਲੀ',
 'ਰਾਗੁ ਧਨਾਸਰੀ',
 'ਰਾਗੁ ਕੇਿਾਰਾ',
 'ਰਾਗੁ ਆਸਾ',
 'ਰਾਗੁ ਿੁਖਾਰੀ',
 'ਰਾਗੁ ਸੋਰਤਿ',
 'ਰਾਗੁ ਨਟ',
 'ਰਾਗੁ ਸਾਰਗ',
 'ਰਾਗੁ ਜੈਿਸਰੀ',
 'ਰਾਗੁ ਕਾਨੜਾ',
 'ਰਾਗੁ ਮਲਾਰ',
 'ਰਾਗੁ ਭੈਰਉ',
 'ਰਾਗੁ ਪ੍ਰਭਾਿੀ',
 'ਰਾਗੁ ਸੂਿੀ']

## Map `final_raag_matches` with `RAAG_MASTER_LIST`

In [49]:
RAAG_MAP = {
'ਤਸਰੀ ਰਾਗੁ':'ਸਿਰੀ ਰਾਗੁ',
'ਰਾਗੁ ਮਾਝ':'ਰਾਗੁ ਮਾਝ',
'ਰਾਗੁ ਗਉੜੀ':'ਰਾਗੁ ਗਉੜੀ',
'ਰਾਗੁ ਆਸਾ':'ਰਾਗੁ ਆਸਾ',
'ਰਾਗੁ ਗੂਜਰੀ':'ਰਾਗੁ ਗੂਜਰੀ',
'ਰਾਗੁ ਿੇਵਗੰਧਾਰੀ':'ਰਾਗੁ ਦੇਵਗੰਧਾਰੀ',
'ਰਾਗੁ ਤਿਿਾਗੜਾ':'ਰਾਗੁ ਬਿਹਾਗੜਾ',
'ਰਾਗੁ ਵਡਿੰਸੁ':'ਰਾਗੁ ਵਡਹੰਸ',
'ਰਾਗੁ ਸੋਰਤਿ':'ਰਾਗੁ ਸੋਰਠਿ',
'ਰਾਗੁ ਧਨਾਸਰੀ':'ਰਾਗੁ ਧਨਾਸਰੀ',
'ਰਾਗੁ ਜੈਿਸਰੀ':'ਰਾਗੁ ਜੈਤਸਰੀ',
'ਰਾਗੁ ਟੋਡੀ': 'ਰਾਗੁ ਟੋਡੀ',
'ਰਾਗੁ ਿੈਰਾੜੀ':'ਰਾਗੁ ਬੈਰਾੜੀ',
'ਰਾਗੁ ਤਿਲੰਗ':'ਰਾਗੁ ਤਿਲੰਗ',
'ਰਾਗੁ ਸੂਿੀ':'ਰਾਗੁ ਸੂਹੀ',
'ਰਾਗੁ ਤਿਲਾਵਲੁ':'ਰਾਗੁ ਬਿਲਾਵਲ' ,
'ਰਾਗੁ ਗੋੋਂਡ':'ਰਾਗੁ ਗੋਂਡ',
'ਰਾਗੁ ਰਾਮਕਲੀ':'ਰਾਗੁ ਰਾਮਕਲੀ',
'ਰਾਗੁ ਨਟ':'ਰਾਗੁ ਨਟ ਨਾਰਾਇਨ',
'ਰਾਗੁ ਮਾਲੀ':'ਰਾਗੁ ਮਾਲੀ ਗਉੜਾ',
'ਰਾਗੁ ਮਾਰੂ':'ਰਾਗੁ ਮਾਰੂ',
'ਰਾਗੁ ਿੁਖਾਰੀ':'ਰਾਗੁ ਤੁਖਾਰੀ',
'ਰਾਗੁ ਕੇਿਾਰਾ':'ਰਾਗੁ ਕੇਦਾਰਾ',
'ਰਾਗੁ ਭੈਰਉ':'ਰਾਗੁ ਭੈਰਉ',
'ਰਾਗੁ ਿਸੰਿੁ':'ਰਾਗੁ ਬਸੰਤੁ',
'ਰਾਗੁ ਸਾਰਗ':'ਰਾਗੁ ਸਾਰਗ',
'ਰਾਗੁ ਮਲਾਰ':'ਰਾਗੁ ਮਲਾਰ',
'ਰਾਗੁ ਕਾਨੜਾ':'ਰਾਗੁ ਕਾਨੜਾ',
'ਰਾਗੁ ਕਤਲਆਨ':'ਰਾਗੁ ਕਲਿਆਨ',
'ਰਾਗੁ ਪ੍ਰਭਾਿੀ':'ਰਾਗੁ ਪ੍ਰਭਾਤੀ',
'ਰਾਗੁ ਜੈਜਾਵੰਿੀ':'ਰਾਗੁ ਜੈਜਾਵੰਤੀ',
}